In [1]:
import re

import xarray as xr
import pandas as pd

from pathlib import Path

In [2]:
BASE = Path("/gws/ssde/j25b/swift/earajr/NWCSAF_archive/netcdf_2018/RDT")

def get_rdt_file(year, month, day, hour, minute, domain="WestAfrica"):
    date = f"{year:04d}{month:02d}{day:02d}"
    time = f"{hour:02d}{minute:02d}00"

    filename = f"S_NWC_RDT-CW_MSG4_{domain}-VISIR_{date}T{time}Z.nc"

    return BASE / date / filename

In [3]:
f = get_rdt_file(2021, 4, 24, 22, 15)
ds = xr.open_dataset(f)

--------------------------------------------------------------------------
detected:

  Local host:            host839
  Device name:           mlx5_bond_0
  Device vendor ID:      0x02c9
  Device vendor part ID: 4127

Default device parameters will be used, which may result in lower
performance.  You can edit any of the files specified by the
btl_openib_device_param_files MCA parameter to set values for your
device.

NOTE: You can turn off this warning by setting the MCA parameter
      btl_openib_warn_no_device_params_found to 0.
--------------------------------------------------------------------------
--------------------------------------------------------------------------
No OpenFabrics connection schemes reported that they were able to be
used on a specific port.  As such, the openib BTL (OpenFabrics
support) will be disabled for this port.

  Local host:           host839
  Local device:         mlx5_bond_0
  Local port:           1
  CPCs attempted:       rdmacm, udcm
-------

In [4]:
ds

<xarray.Dataset> Size: 23MB
Dimensions:             (time: 1, ny: 1200, nx: 1500, recNUM: 369, nlevel: 2,
                         ncontour: 188, nbslice: 81, nbovershoot: 2,
                         nbpttraj: 12, pal01_colors: 256, pal_RGB: 3)
Coordinates:
    lat                 (ny, nx) float32 7MB ...
    lon                 (ny, nx) float32 7MB ...
  * ny                  (ny) float32 5kB 3.684e+06 3.681e+06 ... 8.701e+04
  * nx                  (nx) float32 6kB -2.775e+06 -2.772e+06 ... 1.722e+06
Dimensions without coordinates: time, recNUM, nlevel, ncontour, nbslice,
                                nbovershoot, nbpttraj, pal01_colors, pal_RGB
Data variables: (12/76)
    NbSigCell           (time) float32 4B ...
    NbConvCell          (time) float32 4B ...
    NbCloudCell         (time) float32 4B ...
    NbElecCell          (time) float32 4B ...
    NbHrrCell           (time) float32 4B ...
    MapCellCatType      (ny, nx) float32 7MB ...
    ...                  ...
    TopAreaTraj         (recNUM, nbpttraj) float32 18kB ...
    CoolingRateTraj     (recNUM, nbpttraj) float32 18kB ...
    ExpanRateTraj       (recNUM, nbpttraj) float32 18kB ...
    SpeedTraj           (recNUM, nbpttraj) float32 18kB ...
    DirTraj             (recNUM, nbpttraj) float32 18kB ...
    MapCellCatType_pal  (pal01_colors, pal_RGB) uint8 768B ...
Attributes: (12/45)
    Conventions:                  CF-1.6
    history:                      2021-04-24T22:43:39Z ncas-sat Product Creat...
    institution:                  National Centre For Atmospheric Science
    source:                       NWC/GEO version v2018
    comment:                      Copyright 2021, EUMETSAT, All Rights Reserved
    references:                   http://nwc-saf.eumetsat.int
    ...                           ...
    geospatial_lat_max:           38.54674
    geospatial_lat_min:           0.7869465
    geospatial_lon_max:           20.886372
    geospatial_lon_min:           -36.402287
    product_quality:              100.0
    product_completeness:         100.0

In [5]:
df = pd.DataFrame({
    "lat": ds["LatIncG"].values,
    "lon": ds["LonIncG"].values,
    "area": ds["Surface"][:, 0].values,
    "btmin": ds["BTmin"][:, 0].values,
    "rain": ds["CRainRate"][:, 0].values,
    "speed": ds["MvtSpeed"].values,
    "dir": ds["MvtDirection"].values,
    "conv": ds["ConvType"].values,
})

In [6]:
df

,lat,lon,area,btmin,rain,speed,dir,conv
0,36.729202,-32.399891,5.200000e+08,222.860001,0.3,17.362001,90.0,2.0
1,NaN,NaN,8.700000e+08,222.860001,1.1,17.270000,90.0,2.0
2,NaN,NaN,8.100000e+08,231.190002,1.0,14.798000,78.0,2.0
3,36.195168,-15.428062,9.150000e+08,226.880005,2.5,12.562000,91.0,2.0
4,NaN,NaN,1.300000e+09,228.639999,2.3,15.327001,96.0,2.0
...,...,...,...,...,...,...,...,...
364,7.794872,1.444138,6.616500e+10,189.250000,NaN,10.534000,256.0,2.0
365,NaN,NaN,9.050500e+10,187.179993,NaN,6.294000,270.0,2.0
366,NaN,NaN,1.350000e+08,211.880005,NaN,7.019001,252.0,2.0
367,7.865946,1.666582,1.785000e+09,211.089996,NaN,3.320000,211.0,1.0


In [7]:
from netCDF4 import Dataset

new_data = Dataset("/gws/ssde/j25b/swift/WISER-EWSA/Leeds_RDT/data/20240516/RDT/S_NWC_RDT-CW_MSG3_Africa-VISIR_20240516T154500Z.nc")

In [8]:
import sys

sys.path.insert(1, "/home/users/mendrika/ASNIE/Utilities/Scripts")

from asnie_utils import (
    GeoGrid,
    generate_time_steps,
    field_to_objects
)

from asnie_loaders import (
    load_core,
    load_rdt
)

In [10]:
new_data

<class 'netCDF4._netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    Conventions: CF-1.6
    history: 2024-05-16T16:11:13Z force-nwcsaf Product Created by NWC/GEO v2021.1
2024-05-16T16:11:13Z force-nwcsaf GEO-RDT-CW-v521 2024-05-16T15:45:00Z Africa.cfg safnwc_RDT-CW.cfm
    institution: National Centre for Atmospheric Science
    source: NWC/GEO version v2021.1
    comment: Copyright 2024, EUMETSAT, All Rights Reserved
    references: http://nwc-saf.eumetsat.int
    contact: safnwchd@aemet.es
    summary: Convection Warning Product of the NWC/GEO. Information about significant convective systems from the triggering to mature phase
    keywords: Convection, Convective Clouds/Systems (observed/analyzed)
    keywords_vocabulary: GCMD Science Keywords
    id: S_NWC_RDT-CW_MSG3_Africa-VISIR_20240516T154500Z.nc
    naming_authority: National Centre for Atmospheric Science
    cdm_data_type: Bulletin
    date_created: 2024-05-16T16:11:13Z
    creator_name: National Centre

In [11]:
new_data.variables

{'NbSigCell': <class 'netCDF4._netCDF4.Variable'>
 uint16 NbSigCell(time)
     long_name: Number of encoded significant RDT-CW cloud cells
     scale_factor: 1.0
     add_offset: 0.0
     units: 1
     valid_range: [   0 5000]
     _FillValue: 65535
 unlimited dimensions: 
 current shape = (1,)
 filling on,
 'NbConvCell': <class 'netCDF4._netCDF4.Variable'>
 uint16 NbConvCell(time)
     long_name: Number of convective cloud cells
     scale_factor: 1.0
     add_offset: 0.0
     units: 1
     valid_range: [   0 5000]
     _FillValue: 65535
 unlimited dimensions: 
 current shape = (1,)
 filling on,
 'NbCloudCell': <class 'netCDF4._netCDF4.Variable'>
 uint16 NbCloudCell(time)
     long_name: Number of analyzed and tracked cloud cells
     scale_factor: 1.0
     add_offset: 0.0
     units: 1
     valid_range: [    0 50000]
     _FillValue: 65535
 unlimited dimensions: 
 current shape = (1,)
 filling on,
 'NbElecCell': <class 'netCDF4._netCDF4.Variable'>
 uint16 NbElecCell(time)
     long_n

In [9]:
t = {
    "year":2024,
    "month":7,
    "day":15,
    "hour":14,
    "minute":00
}

load_rdt(t)

IndexError: lat not found in /